# Chatbot tester links (Colab)

Run the cells **in order**, one at a time. Runtime must be **T4 GPU**. After Cell 4 prints three links, send them to Kshitija and leave Cell 5 running.

In [ ]:
# CELL 1 - install Ollama and download the two models (about 5-10 min the first time)
# Before running: Runtime > Change runtime type > T4 GPU
!nvidia-smi | head -n 12
!apt-get install -y -qq zstd pciutils lshw
!curl -fsSL https://ollama.com/install.sh | sh


In [ ]:
# CELL 2 - start the Ollama server in the background and pull the models
import subprocess, time, os, requests
env = dict(os.environ, OLLAMA_MAX_LOADED_MODELS="2", OLLAMA_KEEP_ALIVE="2h")
subprocess.Popen(["ollama", "serve"], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(30):
    try:
        requests.get("http://localhost:11434", timeout=2); break
    except Exception:
        time.sleep(2)
print("Ollama is up")
!ollama pull llama3.1
!ollama pull mistral


In [ ]:
# CELL 3 - install the chat page library
!pip install -q "gradio>=5,<6" requests


**Rasa (optional, between Cell 3 and Cell 4):** the team ran Rasa in Colab too. Open Sneha's notebook, copy the cells that install Rasa and start it (the one running `rasa run --enable-api`) and paste them here, with her trained model folder. Then set `RASA_URL = "http://localhost:5005"` in Cell 4.

In [ ]:
# CELL 4 - the chat apps. Paste Rasa's public URL below (or leave empty to skip Rasa here).
RASA_URL = ""      # if Rasa runs in THIS Colab (see the Rasa note above): "http://localhost:5005". Otherwise a public https address. No slash at the end.

# IMPORTANT: use the SAME settings as the team pipeline (system prompt, temperature) so testers see the same behaviour that was scored.
# The team run used temperature 0.7 and no other changes; check their notebook for a system prompt.
SYSTEM_PROMPT = ""   # leave empty if the team notebook sends no system prompt
OPTIONS = {"temperature": 0.7, "num_ctx": 8192}   # temperature 0.7 matches the scored runs. num_ctx is 8192 (not 32768) so two models fit on a T4; a 6-message chat is far shorter than this

import requests, gradio as gr

OLLAMA_CHAT = "http://localhost:11434/api/chat"

def make_llm_fn(model):
    def fn(message, history):
        msgs = []
        if SYSTEM_PROMPT:
            msgs.append({"role": "system", "content": SYSTEM_PROMPT})
        for m in history:                       # gradio passes this tester's own history
            msgs.append({"role": m["role"], "content": m["content"]})
        msgs.append({"role": "user", "content": message})
        try:
            r = requests.post(OLLAMA_CHAT, json={"model": model, "messages": msgs, "stream": False,
                                                  "keep_alive": "2h", "options": OPTIONS}, timeout=600)
            return r.json()["message"]["content"]
        except Exception as e:
            return f"[error: {e}]"
    return fn

def rasa_fn(message, history, request: gr.Request):
    sender = request.session_hash or "tester"   # one Rasa conversation per tester
    try:
        r = requests.post(RASA_URL + "/webhooks/rest/webhook",
                          json={"sender": sender, "message": message}, timeout=60)
        texts = [x.get("text", "") for x in r.json() if x.get("text")]
        return "\n".join(texts) if texts else "[no response]"
    except Exception as e:
        return f"[error: {e}]"

def make_app(title, fn):
    return gr.ChatInterface(fn, type="messages", title=title,
                            description="Send the 6 messages from the walkthrough page, one at a time. Replies can take several seconds.")

apps = {
    "Chatbot A (Llama 3.1 8B)": make_app("Chatbot A", make_llm_fn("llama3.1")),
    "Chatbot B (Mistral 7B)":   make_app("Chatbot B", make_llm_fn("mistral")),
}
if RASA_URL:
    apps["Chatbot C (Rasa)"] = make_app("Chatbot C", rasa_fn)

links = {}
port = 7861
for name, app in apps.items():
    _, local, share = app.queue().launch(share=True, prevent_thread_lock=True,
                                         server_port=port, quiet=True)
    links[name] = share
    port += 1

print("\n=== SEND THESE LINKS TO KSHITIJA ===")
for k, v in links.items():
    print(k, "->", v)


In [ ]:
# CELL 5 - keep this running while testers use the bots (leave this cell running and keep the Colab tab open)
import time
while True:
    time.sleep(60)
